<a href="https://colab.research.google.com/github/keden49/Machine-learning-with-python-freecode-camp/blob/main/Titanic_Keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
TensorFlow Decision Forests (TF-DF) is a library for training, serving, and interpreting
decision forest models (Random Forests, Gradient Boosted Trees) within the TensorFlow ecosystem.

Key Features:
1. Seamless integration with TensorFlow and Keras
2. Can mix neural networks and decision forests
3. Built-in feature preprocessing
4. Automatic handling of missing values and categorical features
5. No need for extensive data preprocessing
 '''
!pip install tensorflow_decision_forests

In [22]:
'''
pandas provides DataFrame structures for handling tabular data (like CSV files)
Used for loading, cleaning, and preparing the Titanic dataset
'''
import pandas as pd


In [23]:
# Importing TensorFlow 2.x - the main deep learning framework

import tensorflow as tf

In [4]:
# This provides Random Forest and Gradient Boosted Tree algorithms

import tensorflow_decision_forests as tfdf


In [5]:
# Keras is TensorFlow's high-level neural networks API
# Provides simplified interface for building and training models
from tensorflow import keras

In [6]:
# uploading the titanic dataset

from google.colab import files
uploaded = files.upload()



Saving eval.csv to eval.csv
Saving train.csv to train.csv


In [7]:
# printing current files
import os
print(os.listdir())

['.config', 'train.csv', 'eval.csv', 'sample_data']


In [8]:
# Loading the dataset
x_train=pd.read_csv('train.csv')
x_eval=pd.read_csv('eval.csv')

In [9]:
#first few rows of training data
x_train.head()


,survived,sex,age,n_siblings_spouses,parch,fare,class,deck,embark_town,alone
0,0,male,22.0,1,0,7.2500,Third,unknown,Southampton,n
1,1,female,38.0,1,0,71.2833,First,C,Cherbourg,n
2,1,female,26.0,0,0,7.9250,Third,unknown,Southampton,y
3,1,female,35.0,1,0,53.1000,First,C,Southampton,n
4,0,male,28.0,0,0,8.4583,Third,unknown,Queenstown,y


In [10]:
# first few rows of testing data
x_eval.head()

,survived,sex,age,n_siblings_spouses,parch,fare,class,deck,embark_town,alone
0,0,male,35.0,0,0,8.0500,Third,unknown,Southampton,y
1,0,male,54.0,0,0,51.8625,First,E,Southampton,y
2,1,female,58.0,0,0,26.5500,First,C,Southampton,y
3,1,female,55.0,0,0,16.0000,Second,unknown,Southampton,y
4,1,male,34.0,0,0,13.0000,Second,D,Southampton,y


In [12]:
'''
label encoding - converting categories to numbers
inplace modifies the orginal object
it doesnt create a new object with the modified data
'''
x_train = pd.read_csv('train.csv')
x_eval = pd.read_csv('eval.csv')
x_train['sex'].replace(('male', 'female'), (0, 1), inplace=True)
x_eval['sex'].replace(('male', 'female'), (0, 1), inplace=True)

x_train['alone'].replace(('n', 'y'), (0, 1), inplace=True)
x_eval['alone'].replace(('n', 'y'), (0, 1), inplace=True)

x_train['class'].replace(('First', 'Second', 'Third'), (1, 2, 3), inplace=True)
x_eval['class'].replace(('First', 'Second', 'Third'), (1, 2, 3), inplace=True)



/tmp/ipython-input-3361221821.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  x_train['sex'].replace(('male', 'female'), (0, 1), inplace=True)
/tmp/ipython-input-3361221821.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x_train['sex'].replace(('male', 'female'), (0, 1), inplace=True)
/tmp/i

In [13]:
# new string column output
x_train.head()

,survived,sex,age,n_siblings_spouses,parch,fare,class,deck,embark_town,alone
0,0,0,22.0,1,0,7.2500,3,unknown,Southampton,0
1,1,1,38.0,1,0,71.2833,1,C,Cherbourg,0
2,1,1,26.0,0,0,7.9250,3,unknown,Southampton,1
3,1,1,35.0,1,0,53.1000,1,C,Southampton,0
4,0,0,28.0,0,0,8.4583,3,unknown,Queenstown,1


In [14]:
'''
These columns are dropped because:
1. 'embark_town' has high cardinality (many unique values)
2. 'deck' has many missing values
3. These features might not contribute much to prediction
axis = 1 cuts across columns
'''
x_train.drop(['embark_town', 'deck'], axis=1, inplace=True)
x_eval.drop(['embark_town', 'deck'], axis=1, inplace=True)

In [15]:
# Extracting target variables
y_train = x_train.pop('survived')
y_eval = x_eval.pop('survived')

In [16]:
'''
Features after processing
tolist() is used on pandas Series or NumPy arrays to convert them into a Python list
'''
print("Features in x_train:", x_train.columns.tolist())



Features in x_train: ['sex', 'age', 'n_siblings_spouses', 'parch', 'fare', 'class', 'alone']


In [17]:
#shape of dataset

print(x_train.shape)
print(x_eval.shape)

(627, 7)
(264, 7)


In [46]:
# Building Keras Model
num_features = x_train.shape[1] #input_shape
'''
Sequential model is a container that allows you to stack layers on top of each other in a specific order
The output of one layer automatically becomes the input of the next.
1st argument is neuron capacity(decision maker)
Early layers have wider neuron capacity
later layers have condenced neuron capacity to allow final predictions
We use 3 layers as our ds is small
relu allows for backpropagation,non linear activation
Rectified Linear Unit (ReLU) is a piecewise linear function that will output the input directly if it is positive, otherwise, it will output zero.
Sigmoid is used for final decision making
Dense layers are fully connected layers
input_shape is the shape of the input data
'''

model = tf.keras.Sequential([
    # Input layer: takes raw data as tensors
    tf.keras.layers.Dense(64, activation='relu', input_shape=(num_features,)),

    # layer 2: Digging deeper into the patterns, establishes complex relationships
    tf.keras.layers.Dense(16, activation='relu'),

    # Output layer
    tf.keras.layers.Dense(1, activation='sigmoid')
])



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [54]:
'''
The learning algorithm - how the model adjusts its weights
Loss used for binary classification (identifying one of two classes, 0 or 1)
metrics tells you the percentage of passengers the model guessed correctly
'''
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [59]:
'''
Preventing Overfitting
Callbacks intervene, monitor, or modify the training process.
EarlyStopping Stops training early to prevent overfitting
Stops training when a val_loss has stopped improving.
restore_best_weights Restores the model weights from the epoch with the best value of the monitored quantity.
Patience - number of epochs with no improvement after which training will be stopped.
mode "auto" checks if the val_loss quantity should be maximized or minimized.
min_delta - minimum change in the monitored quantity to qualify as an improvement.
'''
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, mode='auto', restore_best_weights=True, verbose=1, min_delta=0.001)

In [56]:
# Saves best intermediate models
from tensorflow.keras.callbacks import ModelCheckpoint

'''
ModelCheckpoint saves the model's weights during training.
filepath: The path and filename where the model will be saved.
monitor: The metric to monitor. The model will only be saved if this val_accuracy improves.
save_best_only: If True, only the best model (based on the monitored quantity) is saved.
'''
model_checkpoint = ModelCheckpoint(filepath='best_model.keras', monitor='val_accuracy', save_best_only=True)

In [57]:
# Save the final trained model(last_epoch)
#Saved on file my_model.keras
model.save('my_model.keras')

In [60]:
'''
TRAINING LOOP - where the model actually learns from data
Train the model
Training data: Used to update weights (model learns from mistakes)
Validation data: Used to check progress (no learning from this!)
'''
print("Training the model...")
train_model = model.fit(
    x_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=(x_eval, y_eval),#Separate data the model hasn't seen during training
    callbacks=[early_stopping, model_checkpoint],
verbose=1)

Training the model...
Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8159 - loss: 0.4519 - val_accuracy: 0.7841 - val_loss: 0.4608
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8041 - loss: 0.4257 - val_accuracy: 0.7765 - val_loss: 0.4767
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8173 - loss: 0.3937 - val_accuracy: 0.7765 - val_loss: 0.4669
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7854 - loss: 0.4570 - val_accuracy: 0.7765 - val_loss: 0.4743
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8124 - loss: 0.4487 - val_accuracy: 0.7614 - val_loss: 0.5039
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8338 - loss: 0.3947 - val_accuracy: 0.7841 - val_loss: 0.4809
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7679 - loss: 0.4663 - val_accuracy: 0.7765 - val_loss: 0.4636
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8474 - loss: 0.3911 - val_accura

In [ ]:
#Model paused training at 11 as it valuation loss didnt improve after 10 consectutive epochs

In [62]:
#using model and loading model
from tensorflow.keras.models import load_model
import numpy as np

loading_model = load_model('my_model.keras')



/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 14 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [63]:
#new data needs to be in eaxct format as training data
x_new_data = x_eval.copy() # Using x_eval as new data

# Make predictions
predictions = loading_model.predict(x_new_data)
print("Predictions for the first 5 entries of the new data:")
for i in range(5):
    # The model outputs a probability (between 0 and 1).
    predicted_class = 'Survived' if predictions[i][0] > 0.5 else 'Did not survive'#predictions[0] is for column index NB has only one column
    print(f"Passenger {i+1}: {predicted_class} (Probability: {predictions[i][0]:.4f})")#:.4f for 4 decimal places

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
Predictions for the first 5 entries of the new data:
Passenger 1: Did not survive (Probability: 0.0952)
Passenger 2: Did not survive (Probability: 0.4114)
Passenger 3: Survived (Probability: 0.7710)
Passenger 4: Survived (Probability: 0.5824)
Passenger 5: Did not survive (Probability: 0.1777)


In [64]:
from sklearn.metrics import accuracy_score
import numpy as np

# predictions looks like: [[0.85], [0.23], [0.67], ...]
# (predictions > 0.5) creates True/False array
# .astype(int) converts True→1, False→0
binary_predictions = (predictions > 0.5).astype(int)

# Calculate accuracy
accuracy = accuracy_score(y_eval, binary_predictions)

print(f"Accuracy of the model on the evaluation set: {accuracy:.4f}")

Accuracy of the model on the evaluation set: 0.7803
